In [1]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
)
import json
import os

#NOTE: CONFIG
TEST_CSV  = r"C:\Users\ASUS\Downloads\CodeKubbb\Idekteppp\academy_alpha-i-week4\data\synthetic_plant_test.csv"
MODEL_IN  = r"C:\Users\ASUS\Downloads\CodeKubbb\Idekteppp\academy_alpha-i-week4\data\xgb_plant_model.json"
META_OUT_JSON = r"C:\Users\ASUS\Downloads\CodeKubbb\Idekteppp\academy_alpha-i-week4\data\test_eval_metadata.json"
META_OUT_CSV  = r"C:\Users\ASUS\Downloads\CodeKubbb\Idekteppp\academy_alpha-i-week4\data\test_eval_metadata.csv"

FEATURES = ["temp_c", "humidity_pct", "lux", "vpd_kpa"]
TARGET = "y"
LABELS = [0, 1, 2]

#NOTE: Load TEST data only
test_df = pd.read_csv(TEST_CSV)
test_df = test_df.dropna(subset=FEATURES + [TARGET])

X_test = test_df[FEATURES].values
y_test = test_df[TARGET].astype(int).values

#NOTE: Load model
model = XGBClassifier()
model.load_model(MODEL_IN)

#NOTE: Evaluate TEST
test_pred = model.predict(X_test)
test_f1 = f1_score(y_test, test_pred, average="macro", labels=LABELS)
test_acc = accuracy_score(y_test, test_pred)
conf_mat = confusion_matrix(y_test, test_pred, labels=LABELS).tolist()
cls_report = classification_report(y_test, test_pred, labels=LABELS, digits=4, output_dict=True)

#NOTE: Save Metadata
metadata = {
    "test_samples": len(y_test),
    "test_class_distribution": test_df[TARGET].value_counts().to_dict(),
    "accuracy": round(test_acc, 4),
    "macro_f1": round(test_f1, 4),
    "confusion_matrix": conf_mat,
    "classification_report": cls_report
}

with open(META_OUT_JSON, "w") as jf:
    json.dump(metadata, jf, indent=4)

pd.DataFrame(cls_report).transpose().to_csv(META_OUT_CSV, index=True)

print("\n=== TEST PERFORMANCE (Never-seen data) ===")
print(f"Accuracy : {test_acc:.4f}")
print(f"Macro-F1 : {test_f1:.4f}")
print("\nConfusion matrix:")
print(conf_mat)
print("\nClassification report:")
print(pd.DataFrame(cls_report).transpose())


=== TEST PERFORMANCE (Never-seen data) ===
Accuracy : 0.9885
Macro-F1 : 0.9603

Confusion matrix:
[[374, 1, 0], [3, 380, 1], [0, 4, 22]]

Classification report:
              precision    recall  f1-score     support
0              0.992042  0.997333  0.994681  375.000000
1              0.987013  0.989583  0.988296  384.000000
2              0.956522  0.846154  0.897959   26.000000
accuracy       0.988535  0.988535  0.988535    0.988535
macro avg      0.978526  0.944357  0.960312  785.000000
weighted avg   0.988406  0.988535  0.988354  785.000000
